# Korean LLM Evaluation Pipeline

End-to-end pipeline that evaluates a Korean LLM model:
1. **Setup** — Verify EvalHub and Korean MCQ provider
2. **Accuracy** — 5 Korean benchmarks (CLIcK, HAE-RAE, KMMLU, KMMLU-HARD, KoBEST BoolQ)
3. **Performance** — GuideLLM throughput benchmark
4. **Reports** — Generate report.html + RESULTS.md
5. **Notify** — Slack webhook notification

## Prerequisites
- RHOAI 3.5 Data Science Pipelines configured (pipeline server ready)
- `kfp` SDK installed (`pip install kfp>=2.14.3`)
- `.env` file with `EVALHUB_URL`, `MODEL_ENDPOINT`, `PIPELINE_API_URL`, `OCP_TOKEN`, etc.

## Step 1: Configuration

In [19]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

# Pipeline server
NAMESPACE        = os.getenv("NAMESPACE", "demo")
PIPELINE_API_URL = os.getenv("PIPELINE_API_URL", "")  # e.g. https://ds-pipeline-dspa.apps.<cluster>
OCP_TOKEN        = os.getenv("OCP_TOKEN", "") or os.getenv("EVALHUB_AUTH_TOKEN", "")

# Model & EvalHub
MODEL_NAME         = os.getenv("MODEL_NAME", "glm-53-flash")
MODEL_ENDPOINT     = os.getenv("MODEL_ENDPOINT", "")
MODEL_API_KEY      = os.getenv("MODEL_API_KEY", "")
EVALHUB_URL        = os.getenv("EVALHUB_URL", "")
EVALHUB_AUTH_TOKEN = os.getenv("EVALHUB_AUTH_TOKEN", "")

# Options
SLACK_WEBHOOK_URL = os.getenv("SLACK_WEBHOOK_URL", "")
LIMIT = int(os.getenv("LIMIT", "10000"))

print(f"Namespace:        {NAMESPACE}")
print(f"Pipeline API URL: {PIPELINE_API_URL or '⚠ not set — add PIPELINE_API_URL to .env'}")
print(f"OCP Token:        {'✅ set' if OCP_TOKEN else '⚠ not set — add OCP_TOKEN to .env'}")
print(f"Model Name:       {MODEL_NAME}")
print(f"Model Endpoint:   {MODEL_ENDPOINT}")
print(f"Model API Key:    {'✅ set' if MODEL_API_KEY else 'not set'}")
print(f"EvalHub URL:      {EVALHUB_URL}")
print(f"Limit:            {LIMIT}")
print(f"Slack Webhook:    {'configured' if SLACK_WEBHOOK_URL else 'not set'}")

Namespace:        demo
Pipeline API URL: https://ds-pipeline-dspa-demo.apps.your-cluster.example.com
OCP Token:        ✅ set
Model Name:       glm-53-flash
Model Endpoint:   https://maas.apps.your-cluster.example.com/prelude-maas/glm-53-flash
Model API Key:    ✅ set
EvalHub URL:      https://evalhub-demo.apps.your-cluster.example.com
Limit:            100
Slack Webhook:    not set


### Pre-flight Check: MLflow Workspace & Korean MCQ Provider

MLflow 3.x requires namespace-level workspace configuration. If these are missing, the pipeline will fail with `"Workspace context is required"`. This cell verifies (and creates if needed):
1. `mlflow-artifact-connection` Secret
2. `MLflowConfig` CR
3. Korean MCQ Evaluation provider registration

> **Tip:** If you already ran `0_setup/1_eval_hub_setup.ipynb`, these should already exist.

In [ ]:
import subprocess, json as _json

def _resource_exists(kind: str, name: str, ns: str) -> bool:
    return subprocess.run(
        ["oc", "get", kind, name, "-n", ns],
        capture_output=True, text=True,
    ).returncode == 0

all_ok = True

# 1. mlflow-artifact-connection Secret
if _resource_exists("secret", "mlflow-artifact-connection", NAMESPACE):
    print(f"\u2705 Secret 'mlflow-artifact-connection' exists")
else:
    s3_vars = {
        "AWS_ACCESS_KEY_ID": os.getenv("AWS_ACCESS_KEY_ID", "admin"),
        "AWS_SECRET_ACCESS_KEY": os.getenv("AWS_SECRET_ACCESS_KEY", "admin"),
        "AWS_S3_BUCKET": os.getenv("AWS_S3_BUCKET", "models"),
        "AWS_S3_ENDPOINT": os.getenv("AWS_S3_ENDPOINT", "http://seaweedfs-s3.model-storage.svc:8333"),
        "AWS_DEFAULT_REGION": os.getenv("AWS_DEFAULT_REGION", "us-east-1"),
    }
    cmd = ["oc", "create", "secret", "generic", "mlflow-artifact-connection", "-n", NAMESPACE]
    for k, v in s3_vars.items():
        cmd.append(f"--from-literal={k}={v}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print(f"\u2705 Created Secret 'mlflow-artifact-connection'")
    else:
        print(f"\u274c Failed to create Secret: {r.stderr.strip()}")
        all_ok = False

# 2. MLflowConfig CR
if _resource_exists("mlflowconfig", "mlflow", NAMESPACE):
    print(f"\u2705 MLflowConfig 'mlflow' exists")
else:
    manifest = {
        "apiVersion": "mlflow.kubeflow.org/v1",
        "kind": "MLflowConfig",
        "metadata": {"name": "mlflow", "namespace": NAMESPACE},
        "spec": {"artifactRootSecret": "mlflow-artifact-connection"},
    }
    r = subprocess.run(
        ["oc", "apply", "-f", "-"],
        input=_json.dumps(manifest), capture_output=True, text=True,
    )
    if r.returncode == 0:
        print(f"\u2705 Created MLflowConfig 'mlflow'")
    else:
        print(f"\u274c Failed to create MLflowConfig: {r.stderr.strip()}")
        all_ok = False

# 3. EvalHub workspace probe
r = subprocess.run(
    ["oc", "logs", "deploy/evalhub", "-c", "evalhub", "-n", NAMESPACE],
    capture_output=True, text=True,
)
if 'workspaces_enabled":true' in r.stdout or 'workspaces_enabled\\":true' in r.stdout:
    print(f"\u2705 EvalHub workspace support enabled")
else:
    print(f"\u26a0\ufe0f  EvalHub restarting to probe MLflow workspace...")
    subprocess.run(["oc", "delete", "pod", "-l", "app=evalhub", "-n", NAMESPACE], capture_output=True)
    import time; time.sleep(15)
    print(f"\u2705 EvalHub restarted")

# 4. Korean MCQ provider
import httpx
try:
    resp = httpx.get(
        f"{EVALHUB_URL}/api/v1/evaluations/providers",
        headers={"Authorization": f"Bearer {EVALHUB_AUTH_TOKEN}", "X-Tenant": NAMESPACE},
        verify=False, timeout=10,
    )
    providers = resp.json().get("items", [])
    korean = [p for p in providers if p.get("name") == "Korean MCQ Evaluation"]
    if korean:
        print(f"\u2705 Korean MCQ provider registered (id={korean[0]['resource']['id']})")
    else:
        print(f"\u274c Korean MCQ provider NOT registered \u2014 run 0_setup/1_eval_hub_setup.ipynb first")
        all_ok = False
except Exception as e:
    print(f"\u274c Cannot reach EvalHub: {e}")
    all_ok = False

print()
if all_ok:
    print("\u2705 All pre-flight checks passed! Proceed to compile and run.")
else:
    print("\u274c Some checks failed. Fix the issues above before running the pipeline.")

## Step 2: Compile Pipeline to YAML

Compile the KFP v2 pipeline definition to YAML format.
This follows RHOAI 3.5 §1.2.1.

In [20]:
from kfp import compiler
from pipeline import eval_pipeline

PIPELINE_YAML = "eval_pipeline.yaml"

compiler.Compiler().compile(
    pipeline_func=eval_pipeline,
    package_path=PIPELINE_YAML,
)
print(f"Pipeline compiled: {PIPELINE_YAML}")

Pipeline compiled: eval_pipeline.yaml


## Step 3: Connect to Pipeline Server

KFP SDK client를 `.env`의 `PIPELINE_API_URL`과 `OCP_TOKEN`으로 인증합니다 (§1.2.3).

In [21]:
assert PIPELINE_API_URL, "PIPELINE_API_URL이 .env에 설정되지 않았습니다."
assert OCP_TOKEN, "OCP_TOKEN (또는 EVALHUB_AUTH_TOKEN)이 .env에 설정되지 않았습니다."

print(f"Connecting to: {PIPELINE_API_URL}")

Connecting to: https://ds-pipeline-dspa-demo.apps.your-cluster.example.com


In [22]:
from kfp.client import Client

client = Client(
    host=PIPELINE_API_URL,
    existing_token=OCP_TOKEN,
    namespace=NAMESPACE,
)

# Verify connection
experiments = client.list_experiments(namespace=NAMESPACE)
print(f"✅ Connected. Experiments: {experiments.total_size or 0}")

✅ Connected. Experiments: 2


## Step 4: Upload Pipeline

Upload the compiled YAML to the pipeline server (§6.3.1).

In [23]:
import json
from datetime import datetime

PIPELINE_NAME = "korean-llm-eval-pipeline"

# Check if pipeline already exists
filter_spec = json.dumps({
    "predicates": [{
        "key": "name",
        "operation": "EQUALS",
        "string_value": PIPELINE_NAME,
    }]
})
existing = client.list_pipelines(filter=filter_spec)
if existing.pipelines:
    pipeline_id = existing.pipelines[0].pipeline_id
    ver_name = f"{PIPELINE_NAME}-{datetime.now().strftime('%m%d-%H%M%S')}"
    print(f"Pipeline exists: {pipeline_id}")
    print(f"Uploading version: {ver_name}")
    ver = client.upload_pipeline_version(
        pipeline_package_path=PIPELINE_YAML,
        pipeline_version_name=ver_name,
        pipeline_id=pipeline_id,
    )
    version_id = ver.pipeline_version_id
else:
    pipeline = client.upload_pipeline(
        pipeline_package_path=PIPELINE_YAML,
        pipeline_name=PIPELINE_NAME,
    )
    pipeline_id = pipeline.pipeline_id
    versions = client.list_pipeline_versions(pipeline_id=pipeline_id)
    version_id = versions.pipeline_versions[0].pipeline_version_id
    print(f"Pipeline uploaded: {pipeline_id}")

print(f"Pipeline ID: {pipeline_id}")
print(f"Version ID:  {version_id}")

Pipeline exists: d70030e7-6cb1-4ebf-8f14-ccb85fa8b922
Uploading version: korean-llm-eval-pipeline-0930-094537


Pipeline ID: d70030e7-6cb1-4ebf-8f14-ccb85fa8b922
Version ID:  4b7cbd74-4f06-4b82-89fd-4b601a43c4a1


## Step 5: Create Experiment & Run Pipeline

Create an MLflow experiment and execute the pipeline (§6.5, §6.6.1).

You can also run this from the **RHOAI Dashboard**:
- Dashboard → Pipelines → Pipeline definitions → korean-llm-eval-pipeline → Create run

In [24]:
from datetime import datetime

EXPERIMENT_NAME = "korean-eval-experiment"
ts = datetime.now().strftime('%m%d-%H%M')
PIPELINE_EXPERIMENT = f"{MODEL_NAME}-pipeline-eval-{ts}"

experiment = client.create_experiment(
    name=EXPERIMENT_NAME,
    namespace=NAMESPACE,
)
print(f"Experiment: {experiment.experiment_id}")

run = client.run_pipeline(
    experiment_id=experiment.experiment_id,
    job_name=f"eval-{MODEL_NAME}",
    pipeline_id=pipeline_id,
    version_id=version_id,
    params={
        "evalhub_url": EVALHUB_URL,
        "auth_token": EVALHUB_AUTH_TOKEN,
        "namespace": NAMESPACE,
        "model_endpoint": MODEL_ENDPOINT,
        "model_name": MODEL_NAME,
        "model_api_key": "model-api-key",  # K8s Secret name (created by 0_setup notebook)
        "experiment_name": PIPELINE_EXPERIMENT,
        "limit": LIMIT,
        "slack_webhook_url": SLACK_WEBHOOK_URL,
    },
    enable_caching=False,
)
print(f"Run submitted: {run.run_id}")
print(f"Experiment name: {PIPELINE_EXPERIMENT}")
print(f"Monitor at: Dashboard → Pipelines → Runs")

Experiment: 6fdb7e3f-3a08-4f13-a45d-ba275f9e3b69


Run submitted: f125254d-edd7-421c-85b8-23699399938d
Experiment name: glm-53-flash-pipeline-eval-0930-0945
Monitor at: Dashboard → Pipelines → Runs


## Step 6: Monitor Run

Poll the pipeline run status (§6.6.2).

In [25]:
import time

TERMINAL_STATES = {"SUCCEEDED", "FAILED", "SKIPPED", "ERROR", "CANCELED"}

print(f"Monitoring run: {run.run_id}")
print("-" * 60)

start = time.time()
while True:
    r = client.get_run(run.run_id)
    state = r.state or "UNKNOWN"
    elapsed = int(time.time() - start)
    print(f"  [{elapsed:>5d}s] {state}")

    if state in TERMINAL_STATES:
        print(f"\nPipeline {state} (elapsed: {elapsed}s)")
        break

    time.sleep(30)

Monitoring run: f125254d-edd7-421c-85b8-23699399938d
------------------------------------------------------------
  [    0s] PENDING
  [   30s] RUNNING
  [   60s] FAILED

Pipeline FAILED (elapsed: 60s)


## Done

After the pipeline completes:
- **report.html** and **RESULTS.md** are available as pipeline artifacts
- **MLflow UI** shows all benchmark results under the pipeline experiment
- **Slack notification** sent (if webhook URL was configured)

To re-run with a different model, change `MODEL_NAME` and `MODEL_ENDPOINT` in `.env` and re-execute Step 5.